# Export AlphaEarth embeddings for Bavaria

This notebook exports the annual AlphaEarth embeddings for Bavaria from Google Earth Engine to Google Drive.

Oe year is roughly **190 GB**. For that reason, the notebook handles one year at a time and splits the export into several GeoTIFF files. 
As free sotrage space in Google Drive is limited to 15 GB, it is necessary to extend storage to 200 GB.

Recommended workflow:

1. Select one year.
2. Start the export.
3. Wait until Earth Engine reports `COMPLETED`.
4. Download all files from Google Drive.
5. Check that the local copy is complete.
6. Delete the files from Google Drive and empty the bin.
7. Move on to the next year.

Not exactly glamorous, but reliable.

## 1. Setup

The Earth Engine Python API must be available in the active environment. The installation line is commented out on purpose so the notebook does not reinstall packages every time it is opened.

In [ ]:
import sys

# Only needed if the Earth Engine API is not installed yet.
# Uncomment, run once, and restart the kernel afterwards if necessary.
# !{sys.executable} -m pip install earthengine-api -q

In [ ]:
import time
from datetime import datetime

import ee

print("Earth Engine API version:", ee.__version__)

## 2. Initialize Earth Engine

Authentication credentials may stay on the server, but Earth Engine still needs to be initialized again whenever the kernel is restarted (!).

In [ ]:
PROJECT_ID = "embeddingstest-480211"

try:
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine initialized.")
except Exception:
    print("No active Earth Engine session found. Starting authentication...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine authenticated and initialized.")

## 3. Check active tasks

This step is optional, but useful before launching another large export. It lists tasks that are still waiting or running, so duplicate exports do not quietly eat the remaining Drive space.

In [ ]:
active_tasks = []

for task_item in ee.batch.Task.list():
    task_status = task_item.status()

    if task_status.get("state") in {"READY", "RUNNING"}:
        active_tasks.append(task_status)

if active_tasks:
    print(f"Active tasks: {len(active_tasks)}")
    print("-" * 60)

    for task_status in active_tasks:
        print("Description:", task_status.get("description"))
        print("State:", task_status.get("state"))
        print("Task ID:", task_status.get("id"))
        print("-" * 60)
else:
    print("No active Earth Engine tasks found.")

### Optional: cancel a task

Paste the task ID below only when a running or queued export really needs to be stopped. Leaving the value empty does nothing, which is the safer default.

In [ ]:
TASK_ID_TO_CANCEL = ""

if TASK_ID_TO_CANCEL:
    matching_tasks = [
        task_item
        for task_item in ee.batch.Task.list()
        if task_item.status().get("id") == TASK_ID_TO_CANCEL
    ]

    if matching_tasks:
        task_to_cancel = matching_tasks[0]
        task_to_cancel.cancel()
        print(f"Cancellation requested for task: {TASK_ID_TO_CANCEL}")
    else:
        print("No task with this ID was found.")
else:
    print("No task ID provided. Nothing cancelled.")

## 4. Export settings

Change `YEAR` for each new export. The remaining values should normally stay unchanged unless the export strategy changes.

In [ ]:
# ------------------------------------------------------------
# MAIN SETTING: CHANGE THE YEAR HERE
# ------------------------------------------------------------

YEAR = 2023

# Google Drive folder used for the export.
DRIVE_FOLDER = "AlphaEarth_Bavaria"

# Shared prefix for all files created for the selected year.
FILE_PREFIX = f"alphaearth_bayern_{YEAR}"

# Each output file contains 4096 x 4096 pixels.
# At 10 m resolution, that is roughly 40.96 x 40.96 km.
#
# With 64 float64 bands, one uncompressed file is about 8 GiB,
# which stays below Earth Engine's 16 GiB limit per file.
FILE_DIMENSIONS = 4096

# Rough storage check before starting a rather large export.
EXPECTED_FREE_DRIVE_GB = 200
ESTIMATED_EXPORT_GB = 142
SAFETY_BUFFER_GB = 5

if EXPECTED_FREE_DRIVE_GB < ESTIMATED_EXPORT_GB + SAFETY_BUFFER_GB:
    raise RuntimeError(
        "The expected free Google Drive storage is probably too small. "
        "One annual export needs roughly 190 GB."
    )

print("Selected year:", YEAR)
print("Google Drive folder:", DRIVE_FOLDER)
print("Output prefix:", FILE_PREFIX)
print("Output tile size:", f"{FILE_DIMENSIONS} x {FILE_DIMENSIONS} pixels")

## 5. Load the area of interest and AlphaEarth data

Bavaria is selected from the GAUL level-1 administrative boundaries. The AlphaEarth collection is then filtered to the selected year and the Bavaria extent.

In [ ]:
# Select Bavaria from the GAUL first-level administrative boundaries.
bavaria = (
    ee.FeatureCollection("FAO/GAUL/2015/level1")
    .filter(ee.Filter.eq("ADM0_NAME", "Germany"))
    .filter(ee.Filter.eq("ADM1_NAME", "Bayern"))
)

# A quick sanity check. Empty geometries have a talent for causing
# confusing errors several cells later.
n_bavaria = bavaria.size().getInfo()

if n_bavaria == 0:
    raise RuntimeError(
        "Bavaria was not found in the selected GAUL boundary dataset."
    )

# Load the annual AlphaEarth embedding collection.
embeddings = ee.ImageCollection(
    "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
)

# Keep only images from the selected year that overlap Bavaria.
annual_collection = (
    embeddings
    .filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
    .filterBounds(bavaria.geometry())
)

n_images = annual_collection.size().getInfo()

print("Bavaria boundary features:", n_bavaria)
print(f"AlphaEarth images for {YEAR}:", n_images)

if n_images == 0:
    raise RuntimeError(
        f"No AlphaEarth images were found for {YEAR}. "
        "Please select an available year."
    )

band_names = annual_collection.first().bandNames().getInfo()

print("Number of bands:", len(band_names))
print("Band names:", band_names)

## 6. Prepare the export

The annual images are mosaicked and clipped to Bavaria. Earth Engine then splits the result into several cloud-optimized GeoTIFFs.

Why `4096 × 4096` pixels? With 64 `float64` bands, a larger `8192 × 8192` file would be around 32 GiB before compression and exceed Earth Engine's 16 GiB limit. 

In [ ]:
# Merge all annual AlphaEarth tiles and clip the result to Bavaria.
image = (
    annual_collection
    .mosaic()
    .clip(bavaria.geometry())
)

# Prepare the export task.
# Nothing is submitted to Earth Engine until task.start() is called.
task = ee.batch.Export.image.toDrive(
    image=image,
    description=f"AlphaEarth_Bayern_{YEAR}",
    folder=DRIVE_FOLDER,
    fileNamePrefix=FILE_PREFIX,
    region=bavaria.geometry(),
    scale=10,
    maxPixels=1e13,

    # Split the large export into smaller, manageable files.
    shardSize=256,
    fileDimensions=FILE_DIMENSIONS,
    skipEmptyTiles=True,

    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True
    }
)

print("Export prepared:")
print("  Year:", YEAR)
print("  Task description:", f"AlphaEarth_Bayern_{YEAR}")
print("  Destination folder:", DRIVE_FOLDER)
print("  Estimated total size: approximately 141–142 GB")
print()
print("Run the next cell once to submit the export.")

## 7. Start the export

Run this cell only once for the selected year. Running it repeatedly creates duplicate tasks with the same output prefix, and Earth Engine is unfortunately very willing to help with that.

In [ ]:
task.start()

status = task.status()

print("Export submitted.")
print("Task:", status.get("description"))
print("State:", status.get("state"))
print("Task ID:", status.get("id"))

## 8. Monitor the export

The cell checks the task once per minute until it finishes, fails, or is cancelled. The Google Drive folder may stay empty while the task is `READY` or `RUNNING`; the files usually appear only after completion.

In [ ]:
POLL_INTERVAL_SECONDS = 60

while True:
    status = task.status()
    state = status.get("state", "UNKNOWN")
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print(
        f"[{now}] "
        f"{status.get('description', 'unknown task')}: {state}"
    )

    if state in {"COMPLETED", "FAILED", "CANCELLED"}:
        break

    time.sleep(POLL_INTERVAL_SECONDS)

print("-" * 60)
print("Final state:", state)

if state == "COMPLETED":
    print(
        f"Export completed. The files should now be available in "
        f"Google Drive folder '{DRIVE_FOLDER}'."
    )
elif state == "FAILED":
    print("Earth Engine error message:")
    print(
        status.get(
            "error_message",
            "No error message was provided."
        )
    )
elif state == "CANCELLED":
    print("The export was cancelled.")

## 9. Check a task again later

Earth Engine keeps running the export even if the notebook is closed or the kernel stops. Paste the task ID from the start cell below to check its current status later.

In [ ]:
TASK_ID = ""

if TASK_ID:
    saved_status = ee.data.getTaskStatus(TASK_ID)[0]

    print("Task:", saved_status.get("description"))
    print("State:", saved_status.get("state"))

    if saved_status.get("error_message"):
        print("Error:", saved_status["error_message"])
else:
    print("No task ID provided.")

## 10. After the export

Once the task has completed:

1. Download all files beginning with `alphaearth_bayern_<YEAR>`.
2. Check that the local copy is complete.
3. Delete the exported files from Google Drive.
4. Empty the Google Drive bin as well; otherwise the storage is still occupied.
5. Change `YEAR` and start the next export.

Failed or cancelled Earth Engine tasks do not create finished Drive files, so they do not need the same cleanup.